# 5Gdata — Dataset Introduction

**Portal:** WARA data portal · **Dataset ID:** 3 · **Organization:** Ericsson · **Category:** metric

This notebook introduces the `5Gdata` dataset: *5G device data relating quality of
service with radio conditions*. It contains the mobile cell information and the
geolocation data collected by a set of test devices during a measurement campaign:
987,805 reports from 11 Android devices (smartphones and emulators) in Sweden,
February to August 2022. Each report carries the serving cell and its radio
conditions (RSRP, RSRQ, SINR, CQI, ...), the device's GPS position and speed, and
the statistics of the pings the device sent to `google.se` (latency and loss).

The dataset is meant for studying how the experienced quality of service relates
to the radio conditions, and for building models that predict QoS from radio
measurements.

In this notebook we

* install the data portal client and load the data,
* inspect the dataset metadata and the available file,
* go through the schema and do a little preparation,
* look at the coverage and the activity of the campaign,
* look at the radio conditions (LTE and 5G NR),
* study the ping QoS (latency and loss) and how it relates to the radio conditions,
* collect the caveats and point to next steps.

## Contents

1. [Setup](#1-setup)
2. [Dataset metadata](#2-dataset-metadata)
3. [Available files](#3-available-files)
4. [Loading data and schema](#4-loading-data-and-schema)
5. [Quick look: coverage and activity](#5-quick-look-coverage-and-activity)
6. [Radio conditions](#6-radio-conditions)
7. [QoS: latency and loss](#7-qos-latency-and-loss)
8. [Caveats and next steps](#8-caveats-and-next-steps)

In [ ]:
%pip install -q git+https://github.com/wara-ops/DataportalClient.git@${DATAPORTAL_CLIENT_VERSION}

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from dataportal import DataportalClient

plt.rcParams['figure.figsize'] = [10, 6]

## 1. Setup

To access the data we use the data portal client together with a personal token:
paste your token inside the quotes in the cell below (see the *PortalBasics*
tutorial if you need to obtain one). If you are in the Jupyter environment
provided by the portal and the environment variable `DATAPORTAL_CLIENT_TOKEN` is
set, you can leave `token` empty instead.

In [ ]:
# paste your token inside the quotes (or leave empty if DATAPORTAL_CLIENT_TOKEN is set)
token = ''
dataset = '5Gdata'
client = DataportalClient(token)

## 2. Dataset metadata

The dataset record on the portal says:

> **ShortInfo:** 5G device data relating quality of service with radio conditions.
> **LongInfo:** Contains mobile cell information and geolocation data collected by a test device during a measurement campaign.

The full record, as returned by the portal (id, organization, category, tags, ...):

In [ ]:
client.fromDataset('5Gdata')
print(client.currentDataset().model_dump_json(indent=2))

## 3. Available files

The dataset consists of a single file: the complete 2022 measurement campaign,
as one parquet file. The list below shows the files in the dataset, with the
time span, the number of entries, and the (compressed) size for each.

In [ ]:
files = client.fromDataset('5Gdata').listFiles()
for f in files:
    print(f)

## 4. Loading data and schema

Load the file into a pandas DataFrame using the `FileID` from the list above.
The file is about 27 MB compressed and holds roughly one million reports, so the
first load takes a moment.

In [ ]:
file_id = 36443  # FileID from the list above
df = client.getData(file_id)
print(df.shape)
df.head()

Each row is one report from one device. The columns fall into a few groups:

| Group        | Columns | Content |
|--------------|---------|---------|
| time         | `time` (index), `mTimeStamp` | UTC time the report was received (index) and the device's own timestamp (ms since epoch) |
| message      | `name`, `tags`, `topic`, `DATA_FORMAT_VERSION` | constant message type `CellMeas`, the MQTT topic (includes the device id), format version 5 |
| device       | `DEVICE_INFO_*` (20 columns), `geohash`, `geohashHiRes`, `geohashLowRes` | static properties of the device (model, manufacturer, ...) and a coarse geohash of the position |
| cell         | `mCellID`, `mCellType`, `mCellMcc`, `mCellMnc`, `mCellPci`, `mCellTac`, `mCellConn`, `mRegOnCell`, `mCellInfoTS`, `mCellSig`, `mEarcfn`, `mBW` | the serving cell: identity, radio access technology (GSM/WCDMA/LTE/NR), operator (MCC/MNC), channel number, bandwidth |
| radio        | `mRsrp`, `mRsrq`, `mRssi`, `mRssnr`, `mCqi`, `mCsiSinr`, `mSsRsrp`, `mSsRsrq`, `mSsSinr`, `mTimingAdvance`, `mLevel`, `mAsuLevel` | signal quality of the serving cell: received power (dBm), signal quality (dB), CQI (1-15); `mSs*` are the 5G NR counterparts |
| location     | `mLocLat`, `mLocLon`, `mLocAcc`, `mLocAlt`, `mSpeed` | GPS position (degrees), horizontal accuracy (m), altitude (m), speed (m/s) |
| QoS          | `mNumPings`, `mPingAvg`, `mPingLoss`, `mPingMax`, `mPingMin`, `mPingAge`, `mPingHost` | ping statistics to `google.se` for the report interval: number of pings, round-trip times (ms), number of lost pings, age of the last ping |
| throughput   | `mDLBitrate`, `mULBitrate`, `mLastReportedDLBitrate`, `mLastReportedULBitrate` | reported throughputs (Mbps); mostly empty in this campaign |

There are a few data-quality issues to be aware of before doing any analysis:

* some numeric columns are stored as *strings* (`mRsrp`, `mRsrq`, `mRssi`,
  `mRssnr`, the `mSs*` columns, and the bitrates),
* the value `2147483647` (INT32_MAX) is used as a "no value" sentinel in many
  columns,
* `mCellID`/`mCellTac` equal to 0 means the device is not registered on a cell,
  and `mCqi` equal to 0 means the CQI was not reported,
* `mCsiSinr` is never valid in this dataset, and `geohashHiRes`/`geohashLowRes`
  are always empty.

Let's look at the types and then prepare the data for analysis.

In [ ]:
df.info(memory_usage='deep')

In [ ]:
# Convert the string-typed numeric columns.
num_str_cols = ['mRsrp', 'mRsrq', 'mRssi', 'mRssnr', 'mSsRsrp', 'mSsRsrq',
                'mSsSinr', 'mDLBitrate', 'mULBitrate',
                'mLastReportedDLBitrate', 'mLastReportedULBitrate']
for c in num_str_cols:
    df[c] = pd.to_numeric(df[c], errors='coerce')

# Replace the INT32_MAX sentinel with NaN in the numeric columns.
num_cols = df.select_dtypes('number').columns
df[num_cols] = df[num_cols].mask(df[num_cols] == 2147483647)

# The device id is also part of the MQTT topic; keep it as a plain column.
df['device'] = df.DEVICE_INFO_android_id
print(f'{df.shape[0]} reports, {df.shape[1]} columns')

## 5. Quick look: coverage and activity

The campaign ran from 9 February to 23 August 2022, but the devices only
reported on about half of the days: they were deployed in waves, in different
parts of Sweden.

In [ ]:
# Where did the reports come from?
loc = df[(df.mLocLat > 0) & (df.mLocLon > 0)]
fig, ax = plt.subplots(figsize=(9, 9))
for ct, color in [('GSM', 'tab:red'), ('WCDMA', 'tab:orange'),
                  ('LTE', 'tab:blue'), ('NR', 'tab:green')]:
    sub = loc[loc.mCellType == ct]
    ax.scatter(sub.mLocLon, sub.mLocLat, s=0.3, alpha=0.6,
               color=color, label=ct)
ax.set_aspect(1)
ax.set_xlabel('longitude')
ax.set_ylabel('latitude')
ax.set_title('Position of the reports, colored by radio access technology')
ax.legend()
plt.tight_layout()
plt.show()

The campaign covers several areas of Sweden: mainly Skåne (the Lund/Malmö
region) and the area around Växjö, with parts of the campaign reaching as far
north as Uppsala. Most of the points are LTE; the 5G NR reports (green) are a
small minority.

In [ ]:
# How active was the campaign, day by day?
daily = df.index.to_series().groupby(df.index.to_series().dt.date).size()
fig, ax = plt.subplots()
ax.bar(daily.index, daily.values, width=1.0, color='tab:blue')
ax.set_ylabel('reports per day')
ax.set_title(f'Activity of the campaign: {len(daily)} days with data '
             f'out of {(df.index.max() - df.index.min()).days + 1}')
plt.xticks(rotation=70)
plt.tight_layout()
plt.show()

Eleven devices took part in the campaign: Samsung and Sony smartphones and
Android emulators. The table below shows the number of reports and the time
span for each. The overwhelming majority of the data comes from a single
Samsung Galaxy S10.

In [ ]:
dev = (df.reset_index()
       .groupby('device')
       .agg(model=('DEVICE_INFO_model', 'first'),
            reports=('time', 'size'),
            first=('time', 'min'),
            last=('time', 'max'))
       .sort_values('reports', ascending=False))
dev

## 6. Radio conditions

The core of the dataset is the radio measurements of the serving cell:

* `mRsrp` — received signal power (dBm); strong is around -70, weak is below
  -120,
* `mRsrq` — received signal quality (dB),
* `mRssnr` — signal-to-noise ratio (dB),
* `mCqi` — channel quality indicator, 1-15, higher is better; 0 means not
  reported,
* `mSsRsrp`/`mSsRsrq`/`mSsSinr` — the counterparts measured on the 5G NR
  synchronization signal; only valid when the device is camped on an NR cell,
* `mTimingAdvance` — the uplink timing offset, which is proportional to the
  distance to the cell (cf. the *R-Traffic* dataset).

First, the breakdown of the campaign by radio access technology:

In [ ]:
df.mCellType.value_counts()

The campaign is dominated by LTE; 5G NR is used for only about 1.5% of the
reports, and the rest is 2G/3G fallback.

Let's zoom in on a continuous stretch (no gap of more than 5 minutes) of about
30 hours from the dominant device, in early July 2022:

In [ ]:
# A continuous ~30 hour stretch from the dominant device.
dev_id = df.device.value_counts().idxmax()
w0, w1 = pd.Timestamp('2022-07-05 14:50'), pd.Timestamp('2022-07-06 20:19')
w = df.loc[w0:w1]
w = w[w.device == dev_id]
print(f'{len(w)} reports, {w.index.min()} to {w.index.max()}')

fig, (ax1, ax2, ax3) = plt.subplots(3, 1, sharex=True, figsize=(11, 8))
ax1.plot(w.index, w.mRsrp, '.', ms=1, alpha=0.5, color='tab:blue')
ax1.set_ylabel('mRsrp (dBm)')
ax1.set_title('Radio conditions over time (LTE stretch, dominant device)')
ax2.plot(w.index, w.mRssnr, '.', ms=1, alpha=0.5, color='tab:orange')
ax2.set_ylabel('mRssnr (dB)')
ax3.plot(w.index, w.mCqi, '.', ms=1, alpha=0.5, color='tab:green')
ax3.set_ylabel('mCqi (1-15)')
plt.tight_layout()
plt.show()

The signal level moves with the device as it travels: `mRsrp` typically varies
between -130 and -70 dBm, the signal-to-noise ratio follows, and the CQI jumps
in discrete steps, tracking the modulation and coding the network assigns to
the device.

The 5G NR corner: for the reports where the device was camped on an NR cell,
the synchronization-signal measurements are available:

In [ ]:
nr = df[(df.mCellType == 'NR') & df.mSsRsrp.notna() & df.mSsSinr.notna()]
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(nr.mSsRsrp, nr.mSsSinr, s=1, alpha=0.4, color='tab:green')
ax.set_xlabel('mSsRsrp (dBm)')
ax.set_ylabel('mSsSinr (dB)')
ax.set_title(f'5G NR reports: SS-RSRP vs SS-SINR ({len(nr)} points)')
plt.tight_layout()
plt.show()

## 7. QoS: latency and loss

The quality of service is measured with pings from the device to `google.se`
(`mPingHost`). Each report summarizes a batch of up to 15 pings
(`mNumPings`): `mPingAvg`/`mPingMax`/`mPingMin` are the round-trip times in ms
and `mPingLoss` is the number of pings lost in the batch.

In [ ]:
# The ping QoS in the same window as the radio conditions above.
fig, (ax1, ax2) = plt.subplots(2, 1, sharex=True, figsize=(11, 6))
ax1.plot(w.index, w.mPingAvg, '.', ms=1, alpha=0.5, color='tab:blue')
ax1.set_ylabel('mPingAvg (ms)')
ax1.set_title('Ping QoS over time (same stretch as above)')
ax2.plot(w.index, w.mPingLoss, '.', ms=1, alpha=0.5, color='tab:red')
ax2.set_ylabel('mPingLoss (count)')
plt.tight_layout()
plt.show()

The median latency is around 60 ms, with occasional spikes, and roughly a tenth
of the reports have at least one lost ping.

Now to the point of the dataset: does the QoS relate to the radio conditions?
First, a simple breakdown by radio access technology:

In [ ]:
tab = (df.assign(lossy=df.mPingLoss > 0)
       .groupby('mCellType', observed=True)
       .agg(reports=('lossy', 'size'),
            loss_fraction=('lossy', 'mean'),
            latency_median_ms=('mPingAvg', 'median')))
tab.round(3)

The strongest effect is the radio access technology: the 2G/3G fallback (GSM,
WCDMA) is much lossier and much slower than LTE. Within LTE, the relation
between the signal level and the QoS is, surprisingly, very weak:

In [ ]:
lte = df[(df.mCellType == 'LTE') & df.mRsrp.notna()]
print(f"corr(mRsrp, mPingAvg) = {lte.mRsrp.corr(lte.mPingAvg):+.3f}")
print(f"corr(mRsrp, loss>0)   = {lte.mRsrp.corr((lte.mPingLoss > 0).astype(int)):+.3f}")

Both correlations are close to zero: in this campaign the internet-side latency
to `google.se` (median around 60 ms) dominates the last radio hop, and it is
the radio access technology rather than the raw signal level that determines
the QoS. A proper study would also account for the cell, the area, the time of
day, and the movement of the device, which is left as an exercise.

## 8. Caveats and next steps

**Caveats**

* **Sentinels and dead columns.** `2147483647` (INT32_MAX) means "no value" in
  many columns; `mCellID`/`mCellTac` equal to 0 means "not on a cell"; `mCqi`
  equal to 0 means "not reported". The `mCsiSinr` column is never valid in this
  dataset, and `geohashHiRes`/`geohashLowRes` are always empty.
* **String-typed numerics.** `mRsrp`, `mRsrq`, `mRssi`, `mRssnr`, the `mSs*`
  columns, and the bitrates are strings and must be converted before use.
* **Two timestamps.** The `time` index is the time the portal received the
  report; the device's own timestamp is `mTimeStamp` (ms since epoch), which is
  usually a little earlier (median about 0.1 s) but deviates by seconds or more
  for roughly a quarter of the rows.
* **Intermittent campaign.** Data exists for 87 of 196 days; the devices were
  deployed in waves and different devices cover different areas.
* **Emulators and bad fixes.** Three of the devices are Android emulators
  (`Kona`/`Lahaina`/`Taro for arm64`); a few thousand rows have no valid GPS
  position (latitude/longitude 0).
* **Imbalanced QoS labels.** About 88% of the reports have no lost pings, so a
  classifier for ping loss must cope with an imbalanced target.
* `mCellConn` is `REQUIRES SDK 28` for about a fifth of the rows: on those
  Android versions some cell measurements could not be obtained.

**Next steps**

* analyse the campaign device by device (the dominant Galaxy S10 is about 78%
  of the data),
* study handovers and cell changes over time (`mCellID`),
* do a 5G NR-specific study using the `mSs*` measurements,
* aggregate the radio and QoS statistics per cell or per area,
* build and evaluate a model that predicts ping loss from the radio conditions
  (mind the imbalanced target).